<a href="https://colab.research.google.com/github/Rini43/Asteroid_Diameter_Prediction/blob/main/Exit_exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Asteroides Diameter Predictor

# Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import joblib
import os

# Read the data

In [ ]:
# filepath to the .excel file
# Load the dataset from the cloned GitHub repository

filepath = '/content/drive/MyDrive/Data/dataset.csv'
df_asteroid = pd.read_csv(filepath)
df_asteroid.head(20)

## EDA

In [ ]:
# Identifying numerical and categorical data
df_asteroid.info()

In [ ]:
# Identifying dataset size
df_asteroid.shape

In [ ]:
# Statistical Summary
df_asteroid.describe()

In [ ]:
df_asteroid.columns.tolist()

In [ ]:
df_asteroid.dtypes

### MIssing Values

In [ ]:
# Trying to findout if there is any missing value present
df_asteroid.isna().sum()

### Duplicated Columns

In [ ]:
df_asteroid.duplicated().sum()

In [ ]:
df_asteroid.columns.tolist()

In [ ]:
# Work on diameter data only
df_diameter = df_asteroid[df_asteroid['diameter'].notna()].copy()

print("Total rows:", len(df_asteroid))
print("Rows with diameter:", len(df_diameter))
print("Missing diameter:", df_asteroid['diameter'].isna().sum())
print(df_diameter['diameter'].describe())

In [ ]:
# Diameter distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(df_diameter['diameter'], bins=80, edgecolor='black', color='skyblue', alpha=0.8)
axes[0, 0].set_title('Distribution of Asteroid Diameter')
axes[0, 0].set_xlabel('Diameter (km)')
axes[0, 0].set_ylabel('Frequency')

axes[0, 1].hist(df_diameter['diameter'], bins=80, edgecolor='black', color='salmon', alpha=0.8)
axes[0, 1].set_yscale('log')
axes[0, 1].set_title('Diameter Distribution (log scale)')
axes[0, 1].set_xlabel('Diameter (km)')
axes[0, 1].set_ylabel('Frequency (log)')

df_diameter['diameter'].plot(kind='density', ax=axes[1, 0], color='green', lw=2)
axes[1, 0].set_title('Density Plot of Diameter')
axes[1, 0].set_xlabel('Diameter (km)')
axes[1, 0].set_ylabel('Density')

axes[1, 1].boxplot(df_diameter['diameter'], patch_artist=True,
                   boxprops=dict(facecolor='lightblue', alpha=0.7),
                   medianprops=dict(color='red'))
axes[1, 1].set_title('Boxplot of Diameter')
axes[1, 1].set_ylabel('Diameter (km)')

plt.tight_layout()
plt.show()

In [ ]:
# Key features to analyze with diameter
selected_features = ['H', 'albedo', 'a', 'e', 'i', 'diameter']
subset = df_asteroid[selected_features].dropna()

subset.head()

In [ ]:
# Correlation heatmap
corr = subset.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Scatterplots: diameter vs selected features
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Diameter vs Selected Features', fontsize=16, fontweight='bold')

feature_list = ['H', 'albedo', 'a', 'e', 'i']
colors = ['blue', 'green', 'red', 'purple', 'orange']

for i, feature in enumerate(feature_list):
    row, col = divmod(i, 3)
    ax = axes[row, col]
    sns.scatterplot(data=subset, x=feature, y='diameter', alpha=0.4, color=colors[i], ax=ax)
    corr_val = subset[[feature, 'diameter']].corr().iloc[0, 1]
    ax.set_title(f'{feature} vs Diameter\n(r = {corr_val:.3f})')
    ax.grid(True, alpha=0.3)

# Hide empty subplot if present
if len(feature_list) < 6:
    axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of important variables
important_vars = ['H', 'albedo', 'a', 'e', 'i']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribution of Important Variables', fontsize=16, fontweight='bold')

for i, var in enumerate(important_vars):
    row, col = divmod(i, 3)
    sns.histplot(df_asteroid[var].dropna(), bins=50, kde=True, ax=axes[row, col], color='steelblue')
    axes[row, col].set_title(f'{var} Distribution')
    axes[row, col].set_xlabel(var)
    axes[row, col].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Optional: diameter categories
bins = [0, 1, 5, 10, 50, 100, 1000]
labels = ['<1 km', '1-5 km', '5-10 km', '10-50 km', '50-100 km', '>100 km']

df_diameter['diameter_category'] = pd.cut(df_diameter['diameter'], bins=bins, labels=labels, right=False)

category_counts = df_diameter['diameter_category'].value_counts().sort_index()
print(category_counts)

plt.figure(figsize=(8, 5))
category_counts.plot(kind='bar', color='mediumseagreen', edgecolor='black')
plt.title('Asteroids by Diameter Category')
plt.xlabel('Diameter Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Summary of correlations with diameter
corr_with_diameter = subset.corr()['diameter'].drop('diameter').sort_values(ascending=False)
print(corr_with_diameter)

# Task 1

In [ ]:
## Question 1

# No, based on the visuals, the target variable “diameter” is not normally distributed.
# It is strongly right-skewed, and a log transformation would make it
# much closer to a normal distribution for modeling purposes.

# The clearest visual evidence is the histogram and Q-Q plot:

# The diameter histogram is strongly right-skewed: most asteroid diameters are concentrated at small values,
# while a long tail extends toward very large diameters.

# The Q-Q plot bends away from the straight reference line, especially in the tails. For a normal distribution,
# the points should stay close to that line.

# The density curve is asymmetric and heavier on the right side, which is another sign of skewness.

# The boxplot shows many outliers above the upper whisker, which is typical of a skewed distribution rather than a normal one.

# This pattern is expected in asteroid datasets because there are many small asteroids and only a few unusually large ones.
# In other words, a small number of very large diameters creates a long right tail.


In [ ]:
# Question 2

# The strongest relationships with asteroid diameter appear to be:

# Absolute magnitude (H): strong negative relationship—larger asteroids generally have lower H values.
# Albedo: a weaker relationship compared with H.
# Orbital features such as semi-major axis (a), eccentricity (e), and inclination (i)
# show relatively weak relationships with diameter.

In [ ]:
# Question 3

# Absolute magnitude (H) remains the strongest predictor, which is fundamentally a
# brightness measurement—more directly related to the physical size and reflectivity of the asteroid.

# Orbital characteristics might influence asteroid size for these reasons:

# Collisional History: Asteroids in different orbital regions experience different collision frequencies and impacts.
# Asteroids in crowded regions (like the inner asteroid belt) undergo more collisions,
# which can fragment larger asteroids into smaller pieces over time.

# Dynamical Stability: Stable orbits allow asteroids to survive longer without being ejected or colliding.
# Asteroids in unstable orbits may be destroyed, leaving only smaller remnants or newly formed small asteroids.

# Migration and Orbital Resonances: Orbital resonances with planets can cause asteroids to migrate into different regions.
# Some regions may be more conducive to retaining larger bodies, while others may preferentially contain smaller asteroids.

# Formation Location: Asteroids formed at different distances from the Sun had different formation conditions
# (temperature, material composition, density). These formation locations correlate with current orbital characteristics,
# which may influence typical asteroid sizes in those regions.

# Compositional Variation by Orbit: Different orbital zones contain asteroids of different compositions
# (C-type, S-type, M-type, etc.). Some compositions may naturally produce larger bodies, or larger bodies of
# certain compositions may survive longer in specific orbits.

# Gravitational Interactions: Close encounters with planets or larger asteroids can alter orbits.
# The orbital characteristics reflect the dynamical history, which is tied to size distribution outcomes.

# However, the weak correlation observed in your data suggests these orbital factors are not primary drivers of asteroid size.


# Data Preprocessing & Feature Engineering

In [ ]:
# Features: orbital and physical properties
# Target: diameter

# Exclude identifier columns and other non-feature columns.
# 'neo', 'pha', and 'class' are intended as features after encoding, so they are not in this exclude list.
exclude_cols = ['id', 'spkid', 'full_name', 'pdes', 'name', 'prefix',
                     'orbit_id', 'equinox', 'epoch_cal']

In [ ]:
# Encode neo and pha (binary Y/N columns)
df_asteroid['neo'] = df_asteroid['neo'].map({'Y': 1, 'N': 0})
df_asteroid['pha'] = df_asteroid['pha'].map({'Y': 1, 'N': 0})

In [ ]:
# One-hot encode class
df_asteroid = pd.get_dummies(df_asteroid, columns=['class'], drop_first=True)

In [ ]:
# First, check what columns exist
print(df_asteroid.columns)

# Check which columns are categorical (object type)
print(df_asteroid.dtypes)

# See unique values in categorical columns
print(df_asteroid.select_dtypes(include='object').head())

In [ ]:
# Drop excluded columns from dataframe

df_asteroid.drop(columns=exclude_cols, inplace=True)

In [ ]:
## Separating Numerical and Categorical data
numeric_cols = df_asteroid.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_asteroid.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

# show dtypes for manual checking
print("\nColumn dtypes:\n", df_asteroid.dtypes)

In [ ]:
df_asteroid.head(5)

In [ ]:
# Remove rows with missing values in key columns
df_clean = df_asteroid.dropna()

print(f"Original dataset: {len(df_asteroid):,} rows")
print(f"After removing missing values: {len(df_clean):,} rows")
print(f"Rows removed: {len(df_asteroid) - len(df_clean):,}")
print(f"Data retention: {len(df_clean)/len(df_asteroid)*100:.2f}%")

# Check remaining missing values
print("\nMissing values after cleaning:")
print(df_clean.isna().sum().sum())

In [ ]:
df_asteroid.isna().sum()

# Train-Test Split

In [ ]:
X = df_asteroid.drop(columns=['diameter'])      # Feature
y = df_asteroid['diameter']                     # Target

In [ ]:
# Split the data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

## Scaling

In [ ]:
X_train.select_dtypes(include='object').columns


In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
y = y.fillna(0)

In [ ]:
# Calculate feature importance

print("="*70)
print("STEP 5: Feature Importance (Random Forest)")
print("="*70)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1, max_depth=20)
rf_model.fit(X, y)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance Ranking:")
for idx, row in feature_importance.iterrows():
    bar = '█' * int(row['importance'] * 100)
    print(f"{row['feature']:25s}: {row['importance']:7.4f} {bar}")

print("\nTop 10 Most Important Features:")
for i, (idx, row) in enumerate(feature_importance.head(10).iterrows(), 1):
    print(f"{i:2d}. {row['feature']:25s} (importance: {row['importance']:.4f})")

STEP 5: Feature Importance (Random Forest)


In [ ]:
# Visualize feature importance

fig, ax = plt.subplots(figsize=(12, 8))

top_20 = feature_importance.head(20)
colors = plt.cm.viridis(np.linspace(0, 1, len(top_20)))
ax.barh(range(len(top_20)), top_20['importance'].values, color=colors)
ax.set_yticks(range(len(top_20)))
ax.set_yticklabels(top_20['feature'].values, fontsize=10)
ax.set_xlabel('Feature Importance', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Most Important Features for Diameter Prediction',
             fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

for i, v in enumerate(top_20['importance'].values):
    ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Task 2

In [ ]:
# Question 1

# Feature scaling is important for DNNs because:

# Faster convergence - Gradient descent updates weights more efficiently
# Prevents vanishing/exploding gradients - Keeps values in safe numerical ranges
# Equal feature contribution - Large-range features don't dominate learning
# One learning rate for all features - Simplifies hyperparameter tuning
# Better weight initialization - Aligns with how weights are initialized
# Fairer regularization - L1/L2 penalties applied equally across features

In [ ]:
# Question 2

# Features Removed:

# Identifiers (7 columns): id, spkid, full_name, pdes, name, prefix, orbit_id

# Why: Non-predictive metadata, no relationship to diameter
# Target Variable (1 column): diameter

# Why: Cannot use target as input feature
# Measurement Uncertainty (1 column): diameter_sigma

# Why: Correlates with data availability, not actual diameter; adds noise
# Redundant Time Variables (3 columns): epoch, epoch_cal, tp_cal

# Why: Epoch_mjd kept; these are duplicate time representations
# Derived Orbital Elements (2 columns): per, per_y

# Why: Orbital period is derived from semi-major axis (a); redundant
# Measurement Noise/Sigma Columns (11 columns): sigma_e, sigma_a, sigma_q,
# sigma_i, sigma_om, sigma_w, sigma_ma, sigma_ad, sigma_n, sigma_tp, sigma_per

# Why: Measurement errors, not predictive features; add noise without
# predictive power
# Total Removed: 25 columns

# Retained: 32 features (H, albedo, orbital elements, neo, pha, moid,
# class encodings, rms, equinox)

In [ ]:
# Qusetion 3

# How Highly Correlated Features Affect Model Learning:

# Multicollinearity Issues

# Weights become unstable and unreliable
# Small changes in data cause large weight swings
# Model becomes fragile and doesn't generalize well
# Redundant Information

# Same information encoded twice wastes model capacity
# Neural network uses extra layers/neurons for correlated features
# Inefficient use of computational resources
# Difficulty in Feature Importance

# Can't determine which correlated feature truly drives predictions
# Feature importance scores become ambiguous
# Hard to interpret model decisions
# Slower Convergence

# Gradient descent takes longer to optimize weights
# Creates a flat valley in the loss surface
# Training becomes slower and less stable
# Overfitting Risk

# Model fits noise in one correlated feature
# Poor generalization to new data
# Validation performance drops
# Regularization Less Effective

# L1/L2 penalties can't efficiently reduce correlated features
# Regularization may remove useful information instead
# Example: If you have both a (semi-major axis) and per (orbital period) —
# they're mathematically related (Kepler's law). Using both is redundant.

## DNN

In [ ]:
# HYPERPARAMETERS

lr = 0.001  # Learning rate
batch_size = 32  # Batch size
num_neurons = 128  # Number of neurons
dropout_rate = 0.3  # Dropout rate
epochs = 5

print("\n" + "="*70)
print("HYPERPARAMETERS")
print("="*70)
print(f"Learning Rate: {lr}")
print(f"Batch Size: {batch_size}")
print(f"Number of Neurons: {num_neurons}")
print(f"Dropout Rate: {dropout_rate}")

In [ ]:
# BUILD DNN MODEL

model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.20),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.20),

    layers.Dense(32, activation='relu'),

    layers.Dense(1, activation='linear')
])

In [ ]:
# Compile model

optimizer = Adam(learning_rate=lr)
model.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=['mae', 'mse']
)

print("\n" + "="*70)
print("MODEL ARCHITECTURE")
print("="*70)
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=[
        'mae'
    ]
)

In [ ]:
# TRAIN MODEL

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=256,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)
print("Training completed!")

In [ ]:
# PLOT TRAINING HISTORY
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('DNN Training Performance: Loss and MAE', fontsize=14, fontweight='bold')

# Loss plot
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2, color='blue')
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2, color='red')
axes[0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Loss (MSE)', fontsize=12, fontweight='bold')
axes[0].set_title('Training vs Validation Loss', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# MAE plot
axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2, color='green')
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2, color='orange')
axes[1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1].set_ylabel('MAE (km)', fontsize=12, fontweight='bold')
axes[1].set_title('Training vs Validation MAE', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Task 3

In [ ]:
# Question 1

# MOdel Evaluation & Error Analysis

In [ ]:
y_pred = model.predict(X_test).flatten()

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE :", mse)
print("RMSE:", rmse)
print("MAE :", mae)
print("R²  :", r2)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(
    history.history['loss'],
    label='Training Loss'
)
plt.plot(
    history.history['val_loss'],
    label='Validation Loss'
)

plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('DNN Training vs Validation Loss')

plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8, 8))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.3
)

plt.xlabel('Actual Diameter')
plt.ylabel('Predicted Diameter')
plt.title('Actual vs Predicted Asteroid Diameter')

# Perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    'r--'
)

plt.show()


In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6
)


In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=256,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


In [ ]:
y_pred = model.predict(X_test).flatten()

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE :", mse)
print("RMSE:", rmse)
print("MAE :", mae)
print("R²  :", r2)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('DNN Training vs Validation Loss')

plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8, 8))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.3
)

plt.xlabel('Actual Diameter')
plt.ylabel('Predicted Diameter')
plt.title('Actual vs Predicted Asteroid Diameter')

# Perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    'r--'
)

plt.show()
